# 01 — Calibrate a detector from scratch

Run `midas-auto-calibrate` against the CeO₂ Pilatus calibrant frame shipped
inside the wheel. No external data needed — `pip install midas-auto-calibrate`
is the only prerequisite (the `MIDASCalibrant` C binary ships inside the wheel).

**What you'll get out:**
- A refined `DetectorGeometry` (Lsd, beam center, tilts, 15-parameter distortion).
- `pseudo_strain` in microstrain — lower = better calibration.
- `convergence_history` table (per-iteration MeanStrain and geometry).

In [1]:
import shutil, tempfile
from pathlib import Path

import midas_auto_calibrate as mac

print('package version:', mac.__version__)
print('bundled CeO₂ frame:', mac.data.CEO2_PILATUS)
print('shipped binary:', mac.midas_bin('MIDASCalibrant'))

package version: 0.1.0
bundled CeO₂ frame: /Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/midas_auto_calibrate/_data/CeO2_Pilatus.tif
shipped binary: /Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/midas_auto_calibrate/_bin/MIDASCalibrant


## Stage the bundled data into a writable workdir

MIDASCalibrant writes its output CSVs next to the input image, so we copy the
wheel-installed (read-only) data into a tmp dir first.

In [2]:
workdir = Path(tempfile.mkdtemp(prefix='mac_nb_'))
img = workdir / 'CeO2_00001.tif'
shutil.copy(mac.data.CEO2_PILATUS, img)
shutil.copy(mac.data.CEO2_PILATUS_DARK, workdir / 'dark.tif')
shutil.copy(mac.data.CEO2_PILATUS_MASK, workdir / 'mask_upd.tif')
print('workdir:', workdir)

workdir: /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/mac_nb_e2kyctr5


## Run the calibration

In [3]:
config = mac.CalibrationConfig(
    material='CeO2',
    lattice_params=(5.4116, 5.4116, 5.4116, 90, 90, 90),
    wavelength=0.172973,   # Å
    pixel_size=172.0,      # µm
    lsd=657_436.9,         # µm starting guess
    ybc=685.5, zbc=921.0,
    nr_pixels_y=1475, nr_pixels_z=1679,
    dark_file='dark.tif',        # relative to work_dir
    mask_file='mask_upd.tif',
    im_trans_opt=[2],            # invert Z axis (Pilatus convention)
    n_iterations=5,
)

result = mac.auto_calibrate(config, img, work_dir=workdir, n_cpus=4)

print(f'pseudo-strain: {result.pseudo_strain:.2f} ± {result.pseudo_strain_std:.2f} µε')
print(f'Lsd:  {result.geometry.lsd:.1f} µm')
print(f'BC:   ({result.geometry.ybc:.2f}, {result.geometry.zbc:.2f}) px')
print(f'ty:   {result.geometry.ty:+.5f}°')
print(f'tz:   {result.geometry.tz:+.5f}°')

pseudo-strain: 2151.37 ± 2322.35 µε
Lsd:  657436.9 µm
BC:   (685.50, 921.00) px
ty:   +0.00000°
tz:   +0.00000°


## Save the refined geometry

JSON is the lossless native format. `to_midas_params()` writes a MIDAS-native
Parameters.txt that the C binaries consume directly.

In [4]:
result.geometry.to_json(workdir / 'calibration.json')
result.geometry.to_midas_params(workdir / 'calibration_Parameters.txt')
print('files written in', workdir)
for f in sorted(workdir.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size:>9} bytes)')

files written in /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/mac_nb_e2kyctr5
  CeO2_00001.tif  (  9908551 bytes)
  CeO2_00001.tif.checkpoint.txt  (      507 bytes)
  CeO2_00001.tif.convergence_history.csv  (     1657 bytes)
  CeO2_00001.tif.corr.csv  (     3792 bytes)
  CeO2_00001.tif.m_step_trace_iter1.csv  (      411 bytes)
  CeO2_00001.tif.m_step_trace_iter2.csv  (      411 bytes)
  CeO2_00001.tif.m_step_trace_iter3.csv  (      411 bytes)
  CeO2_00001.tif.m_step_trace_iter4.csv  (      411 bytes)
  CeO2_00001.tif.m_step_trace_iter5.csv  (      411 bytes)
  Parameters.txt  (      478 bytes)
  calibrant.stdout  (     4978 bytes)
  calibration.json  (      458 bytes)
  calibration_Parameters.txt  (      232 bytes)
  ci_profiles.csv  ( 13148337 bytes)
  dark.tif  (  9908575 bytes)
  hkls.csv  (    33016 bytes)
  mask_upd.tif  (  2476781 bytes)


## Peek at the convergence trace

See `02_inspect_results.ipynb` for the full static-plot bundle (rings overlay,
residual heatmap, Fourier harmonics, distortion field).

In [5]:
if result.convergence_history:
    for row in result.convergence_history:
        print(f"iter {int(row['Iter']):>2}: MeanStrain {row['MeanStrain_ppm']:>8.3f} µε")
else:
    print('no convergence history (n_iterations must be >= 1 to produce one)')

iter  1: MeanStrain 2151.368 µε
iter  2: MeanStrain 2151.368 µε
iter  3: MeanStrain 2151.368 µε
iter  4: MeanStrain 2151.368 µε
iter  5: MeanStrain 2151.368 µε
